# TerraSpectra — Day 2: Ingest Salinas Hyperspectral Data

## 1. Objective & Prerequisites
- **Objective**: Load, validate, and inspect the Salinas AVIRIS hyperspectral scene (`Salinas_corrected.mat`) and ground-truth labels (`Salinas_gt.mat`).
- **Prerequisites**: Python virtual environment configured (Day 1) with `scipy`, `numpy`, and `terraspectra` installed.

## 2. Core Concepts
- **Hyperspectral Cube**: Unlike standard RGB images with 3 broad spectral bands, hyperspectral imagery captures hundreds of contiguous, narrow spectral bands (e.g. 204 bands in Salinas) spanning 400 nm to 2500 nm.
- **Data Dimensions**: A hyperspectral cube is arranged as $(H, W, C)$ where $H$ is the along-track spatial rows, $W$ is cross-track spatial columns, and $C$ is the spectral band index.
- **MATLAB `.mat` Format**: Scientific benchmark datasets often distribute cubes as `.mat` files, requiring `scipy.io.loadmat` to extract the underlying numeric arrays.

In [1]:
# Import reusable loader functionality from src/terraspectra
import numpy as np
from pathlib import Path
from terraspectra.data.loader import SalinasLoader, SALINAS_CLASS_NAMES

# Initialize loader targeting the raw data directory
loader = SalinasLoader(raw_data_dir='../data/raw')
print('Image Path:', loader.image_path.resolve())
print('Ground-Truth Path:', loader.gt_path.resolve())

Image Path: C:\Users\abhis\Desktop\Tetraspectra\data\raw\Salinas_corrected.mat
Ground-Truth Path: C:\Users\abhis\Desktop\Tetraspectra\data\raw\Salinas_gt.mat


In [2]:
# Load and validate dataset (auto-downloads if missing, or validates local files)
dataset = loader.load(auto_download=True, allow_synthetic_fallback=True)
meta = dataset.metadata

print('--- Dataset Ingestion Summary ---')
print(f'Is Synthetic:        {meta.is_synthetic}')
print(f'Spatial Rows (H):    {dataset.rows}')
print(f'Spatial Cols (W):    {dataset.cols}')
print(f'Spectral Bands (C):  {dataset.bands}')
print(f'Cube Array Shape:    {dataset.cube.shape}')
print(f'Ground-Truth Shape:  {dataset.gt.shape}')
print(f'Cube Data Type:      {dataset.cube.dtype}')
print(f'Memory Consumption:  {meta.memory_usage_mb:.2f} MB')

--- Dataset Ingestion Summary ---
Is Synthetic:        False
Spatial Rows (H):    512
Spatial Cols (W):    217
Spectral Bands (C):  204
Cube Array Shape:    (512, 217, 204)
Ground-Truth Shape:  (512, 217)
Cube Data Type:      int16
Memory Consumption:  43.34 MB


### Observation on Array Shape & Memory
- The cube has shape `(512, 217, 204)` containing $512 \times 217 = 111,104$ individual spatial pixels.
- Each spatial pixel is a 204-dimensional spectral vector.
- Total memory footprint is approximately 43.34 MB, allowing it to fit comfortably in RAM.

In [3]:
# Inspect ground-truth agricultural classes
unique_classes = np.unique(dataset.gt)
print(f'Detected {len(unique_classes)} unique classes:')
for cls_id in unique_classes:
    count = int(np.sum(dataset.gt == cls_id))
    cname = SALINAS_CLASS_NAMES[cls_id] if cls_id < len(SALINAS_CLASS_NAMES) else f'Class_{cls_id}'
    pct = (count / dataset.gt.size) * 100.0
    print(f'  Class {cls_id:2d}: {cname:<26} -> {count:6d} pixels ({pct:5.2f}%)')

Detected 17 unique classes:
  Class  0: Unclassified               ->  56975 pixels (51.28%)
  Class  1: Broccoli_green_weeds_1     ->   2009 pixels ( 1.81%)
  Class  2: Broccoli_green_weeds_2     ->   3726 pixels ( 3.35%)
  Class  3: Fallow                     ->   1976 pixels ( 1.78%)
  Class  4: Fallow_rough_plow          ->   1394 pixels ( 1.25%)
  Class  5: Fallow_smooth              ->   2678 pixels ( 2.41%)
  Class  6: Stubble                    ->   3959 pixels ( 3.56%)
  Class  7: Celery                     ->   3579 pixels ( 3.22%)
  Class  8: Grapes_untrained           ->  11271 pixels (10.14%)
  Class  9: Soil_vinyard_develop       ->   6203 pixels ( 5.58%)
  Class 10: Corn_senesced_green_weeds  ->   3278 pixels ( 2.95%)
  Class 11: Lettuce_romaine_4wk        ->   1068 pixels ( 0.96%)
  Class 12: Lettuce_romaine_5wk        ->   1927 pixels ( 1.73%)
  Class 13: Lettuce_romaine_6wk        ->    916 pixels ( 0.82%)
  Class 14: Lettuce_romaine_7wk        ->   1070 pixels ( 0.96

In [4]:
# Validation test: check that dimension mismatch is caught reliably
fake_cube = np.zeros((100, 100, 10), dtype=np.float32)
fake_gt = np.zeros((120, 100), dtype=np.int64)
try:
    loader.validate(fake_cube, fake_gt, 'fake_cube', 'fake_gt')
except ValueError as e:
    print('Validation successfully caught dimension mismatch:')
    print(f'  --> {e}')

Validation successfully caught dimension mismatch:
  --> Spatial dimension mismatch: cube spatial dimensions (100, 100) do not match ground-truth dimensions (120, 100)


## 3. What I Learned & Common Mistakes
1. **Coordinate Systems**: Salinas `.mat` data stores raw sensor array indices (`row, col`). Do not mistake row/col indices for geographic latitude/longitude.
2. **Missing Band Channels**: Corrected Salinas has 204 bands instead of AVIRIS's raw 224 bands because 20 water-absorption bands (108-112, 154-167, 224) were removed during atmospheric correction.
3. **Data Types**: The raw cube uses 16-bit integers (`int16`). For numerical modeling, convert to `float32` to avoid integer overflow.

### Interview Question
*Q: Why can't we load Salinas directly with Rasterio or GDAL?*
*A: Rasterio and GDAL require geospatial metadata (GeoTIFF tags, CRS EPSG code, affine geotransform). Standard Salinas MATLAB `.mat` files contain only raw numerical matrices without affine or geospatial headers.*

### Exercise
Calculate how many total numerical values (floats/integers) are contained in the entire cube: $512 \times 217 \times 204 = 22,665,216$ values.

## 4. Verification & Milestone Status
- [x] Verified existence of Salinas files in `data/raw/`.
- [x] Confirmed dimensions `(512, 217, 204)` and ground-truth `(512, 217)`.
- [x] Validated class labels and memory footprint (~43.34 MB).
- [x] Reusable loader tested in `tests/test_loader.py`.